# Учебное исследование: Павловск (Пушкинский район, Санкт-Петербург)

Ноутбук загружает данные, готовит территорию и считает показатели городской среды с помощью библиотеки **BlocksNet 1.0.0.a9**. Набор показателей повторяет официальный пример `examples/pipeline.ipynb`.

**Источник данных.** Публичные серверы OpenStreetMap (Overpass) перегружены и блокируют большие автоматические запросы. Поэтому используются **официальные готовые данные BlocksNet по Санкт-Петербургу** (репозиторий `IDUclub/blocksnet-data`, релиз `v0.0.1`). В них уже есть кварталы, население, школы и автомобильный граф. Мы скачиваем архив **один раз** и фильтруем кварталы по границе Павловска.

**Что делает ноутбук:**
1. Определяет границу Павловска (Nominatim, точно по OSM-ID).
2. Скачивает официальный архив BlocksNet (~386 МБ) и распаковывает нужные файлы.
3. Оставляет только кварталы Павловска.
4. Загружает готовый автомобильный граф и считает время в пути между кварталами (минуты).
5. Считает показатели: плотность, морфотипы, доступность территории, обеспеченность школами.
6. Сохраняет результаты в GeoJSON/CSV и проверяет их.

Все данные и результаты складываются в папку `pavlovsk_study_data` рядом с ноутбуком. Если файл уже есть, он переиспользуется.

## Как запускать ноутбук (Windows)

1. Откройте проект в VS Code, выберите интерпретатор из папки `.venv` (Python 3.11).
2. Откройте этот файл и выберите ядро **Python 3 (.venv)**.
3. Запускайте ячейки **строго сверху вниз**: `Shift + Enter`. Можно сразу **Run All Cells** — порядок тогда гарантирован.

Если BlocksNet ещё не установлен (из папки проекта):

```
py -3.11 -m venv .venv
.venv\Scripts\activate
python -m pip install -e ".[study]"
```

Группа зависимостей `study` уже описана в `pyproject.toml`.

> ⚠️ Первый запуск скачивает архив ~386 МБ, это может занять время. Повторные запуски используют уже скачанные файлы.

In [ ]:
import json
import pickle
import shutil
import sys
import urllib.request
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

import blocksnet
import iduedu

warnings.filterwarnings('ignore', category=FutureWarning)

# OSMNX используется только для получения границы через Nominatim.
# Служебный кэш не создаём: граница и результаты сохраняются в pavlovsk_study_data.
ox.settings.use_cache = False
ox.settings.requests_timeout = 120

# Папка для скачанных данных и результатов
DATA_DIR = Path('pavlovsk_study_data')
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / '.gitignore').write_text('*\n', encoding='utf-8')

# Перезагружать данные заново, даже если файлы уже есть
FORCE_RELOAD = False

# Список показателей согласован: считаем все показатели из pipeline.ipynb
CONFIRMED_TO_CONTINUE = True

print('Python    :', sys.version.split()[0])
print('geopandas :', gpd.__version__)
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('osmnx     :', ox.__version__)
print('networkx  :', nx.__version__)
print('blocksnet :', blocksnet.__version__)
print('iduedu    :', iduedu.__version__)
print('Папка данных:', DATA_DIR.resolve())

### Шаг 1. Вспомогательные функции

- `load_or_build_geojson` — читает слой из файла, если он уже есть, иначе создаёт и сохраняет (GeoJSON, WGS84).
- `download_dataset` — скачивает официальный архив BlocksNet один раз; повторно использует уже скачанный.
- `ensure_member` — распаковывает из архива только нужный файл.

Архив большой (~386 МБ), поэтому скачивание разбито на попытки и проверяется целостность файла.

In [ ]:
def load_or_build_geojson(filename, builder, force=False):
    """Прочитать GeoJSON из папки данных или создать его и сохранить (в WGS84)."""
    path = DATA_DIR / filename
    if path.exists() and not (force or FORCE_RELOAD):
        gdf = gpd.read_file(path)
        print(f'[из файла] {filename}: {len(gdf)} объектов')
        return gdf
    gdf = builder()
    gdf = gdf.to_crs(4326)
    gdf.to_file(path, driver='GeoJSON')
    print(f'[создано]  {filename}: {len(gdf)} объектов сохранено')
    return gdf


DATASET_URL = 'https://github.com/IDUclub/blocksnet-data/releases/download/v0.0.1/saint_petersburg.zip'
ZIP_NAME = 'saint_petersburg.zip'
ZIP_PATH = DATA_DIR / ZIP_NAME
MIN_ZIP_SIZE = 300 * 1024 * 1024  # архив весит около 386 МБ


def is_valid_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            return len(archive.namelist()) > 0
    except Exception:
        return False


def download_dataset():
    if ZIP_PATH.exists() and ZIP_PATH.stat().st_size >= MIN_ZIP_SIZE and is_valid_zip(ZIP_PATH):
        print('[из файла] архив уже скачан:', ZIP_PATH)
        return ZIP_PATH
    for attempt in range(1, 6):
        try:
            print(f'Скачивание архива ~386 МБ (попытка {attempt} из 5)...')
            with urllib.request.urlopen(DATASET_URL, timeout=120) as response, open(ZIP_PATH, 'wb') as out:
                next_mark = 50 * 1024 * 1024
                downloaded = 0
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    downloaded += len(chunk)
                    if downloaded >= next_mark:
                        print('  ... скачано, МБ:', round(downloaded / 1024 / 1024))
                        next_mark += 50 * 1024 * 1024
            size = ZIP_PATH.stat().st_size
            print('  скачано, МБ:', round(size / 1024 / 1024, 1))
            if size >= MIN_ZIP_SIZE and is_valid_zip(ZIP_PATH):
                return ZIP_PATH
            print('  файл неполный или повреждён, повторяем')
        except Exception as error:
            print('  ошибка скачивания:', type(error).__name__)
    raise RuntimeError('Не удалось скачать архив. Скачайте его вручную и положите сюда: ' + str(ZIP_PATH) + '\n' + DATASET_URL)


def ensure_member(member, target_name=None):
    """Распаковать один файл из архива в папку данных под нужным именем."""
    target = DATA_DIR / (target_name or Path(member).name)
    if target.exists() and not FORCE_RELOAD:
        print('[из файла]', target.name)
        return target
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = [n for n in archive.namelist() if n.endswith(member)]
        if not names:
            raise FileNotFoundError(f'В архиве нет файла {member}')
        # Распаковываем потоком прямо в нужный файл, не создавая временных копий
        with archive.open(names[0]) as source, open(target, 'wb') as destination:
            shutil.copyfileobj(source, destination)
    print('[распаковано]', target.name)
    return target

### Шаг 2. Граница территории

Территория — **Павловск**, город в Пушкинском районе Санкт-Петербурга (объект OpenStreetMap relation **359180**).

Граница берётся через **Nominatim** — поиск по OSM-ID (самый точный способ), с резервным поиском по названию. Сервер Overpass здесь не используется.

Для расчётов площади и расстояний координаты переводятся в метрическую проекцию (`UTM`).

In [ ]:
OSM_RELATION_ID = 359180
OSM_NAME_QUERIES = [
    {'city': 'Pavlovsk', 'state': 'Saint Petersburg', 'country': 'Russia'},
    'Pavlovsk, Saint Petersburg, Russia',
]


def boundary_from_osmid():
    # Берём объект строго по OSM-ID через Nominatim lookup
    candidates = ox.geocode_to_gdf(f'R{OSM_RELATION_ID}', by_osmid=True)
    polygonal = candidates[candidates.geom_type.isin(['Polygon', 'MultiPolygon'])]
    if len(polygonal) == 0:
        raise RuntimeError('Nominatim lookup не вернул полигон для R359180')
    return polygonal.geometry.iloc[0]


def boundary_from_name():
    for query in OSM_NAME_QUERIES:
        try:
            candidates = ox.geocode_to_gdf(query)
        except Exception as error:
            print('  Nominatim не ответил:', type(error).__name__)
            continue
        polygonal = candidates[candidates.geom_type.isin(['Polygon', 'MultiPolygon'])]
        if len(polygonal) > 0:
            return polygonal.geometry.iloc[0]
    raise RuntimeError('Nominatim не вернул полигон границы')


def build_boundary():
    for method_name, method in [('по OSM-ID', boundary_from_osmid), ('по названию', boundary_from_name)]:
        try:
            geometry = method()
            print('  граница получена. Способ:', method_name)
            return gpd.GeoDataFrame(geometry=[geometry], crs=4326)
        except Exception as error:
            print('  способ не сработал:', method_name, '->', type(error).__name__)
    raise RuntimeError('Не удалось получить границу Павловска')


boundary_4326 = load_or_build_geojson('boundary.geojson', build_boundary)
crs = boundary_4326.estimate_utm_crs()
boundary = boundary_4326.to_crs(crs)
print('Проекция для расчётов:', crs)
print('Тип геометрии границы:', boundary_4326.geom_type.iloc[0])

ax = boundary.plot(figsize=(8, 8), facecolor='none', edgecolor='black')
ax.set_title('Павловск (Пушкинский район, Санкт-Петербург)')
plt.show()

### Шаг 3. Официальные данные BlocksNet

**Источник:** `IDUclub/blocksnet-data`, релиз `v0.0.1`, файл `saint_petersburg.zip` (~386 МБ).

Внутри архива:
- `blocks.geojson` — кварталы всего Санкт-Петербурга, уже подготовленные авторами BlocksNet: назначение территории (`land_use`), население (`population`), площади застройки и **ёмкость школ** (`capacity_school`);
- `graph_drive.pickle` — готовый автомобильный граф (время в пути в минутах, атрибут `time_min`);
- готовые матрицы доступности (мы их не используем — считаем свою для кварталов Павловска).

> ℹ️ Население в наборе — оценка BlocksNet по площади жилья из OSM (примерно 20 кв.м на человека). Это не официальная статистика.

> Если автоматическое скачивание не удаётся, скачайте архив вручную по ссылке и положите его в `pavlovsk_study_data/saint_petersburg.zip`.

In [ ]:
zip_path = download_dataset()
blocks_city_path = ensure_member('blocks.geojson', 'blocks_city.geojson')
graph_path = ensure_member('graph_drive.pickle')

print('Готовые файлы:')
print(' -', blocks_city_path)
print(' -', graph_path)

### Шаг 4. Кварталы Павловска

Загружаем кварталы всего города и оставляем только те, чья **характерная точка** (representative point) попадает внутрь границы Павловска. Так мы не обрезаем кварталы по границе и не искажаем их площадь и население.

Каждому кварталу присваиваем постоянный номер `block_id` и сохраняем результат в `blocks.geojson`.

In [ ]:
blocks_city = gpd.read_file(blocks_city_path)
print('Кварталов во всём наборе (Санкт-Петербург):', len(blocks_city))
print('Доступные колонки:', sorted(blocks_city.columns))

boundary_geom = boundary.union_all()

blocks_city = blocks_city.to_crs(crs)
blocks_city = blocks_city[blocks_city.geometry.notna()].copy()
blocks_city.geometry = blocks_city.make_valid()
# Кварталы в наборе уже нарезаны. Не explode-им их повторно,
# чтобы не раздвоить население и площади между частями мультиполигона.
blocks_city = blocks_city[blocks_city.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)

inside = blocks_city.representative_point().within(boundary_geom)
blocks = blocks_city[inside].reset_index(drop=True)

REQUIRED_COLUMNS = [
    'land_use', 'site_area', 'population',
    'footprint_area', 'build_floor_area', 'living_area', 'non_living_area',
    'capacity_school',
]
missing = [c for c in REQUIRED_COLUMNS if c not in blocks.columns]
assert not missing, f'В готовых кварталах нет колонок {missing}. Есть: {sorted(blocks.columns)}'

# Технические осколки геометрии (площадь меньше 10 кв.м) исключаем:
# иначе деление на почти нулевую площадь даёт огромные выбросы в плотности
MIN_SITE_AREA = 10.0
invalid_area = int((blocks['site_area'].isna() | (blocks['site_area'] < MIN_SITE_AREA)).sum())
if invalid_area:
    print(f'Кварталов-осколков (площадь < {MIN_SITE_AREA:g} кв.м) удалено:', invalid_area)
    blocks = blocks[blocks['site_area'].notna() & (blocks['site_area'] >= MIN_SITE_AREA)].reset_index(drop=True)

# Название назначения территории приходит в виде 'LandUse.RESIDENTIAL'
blocks['land_use'] = (
    blocks['land_use'].astype('string')
    .str.replace('LandUse.', '', regex=False)
    .str.lower()
)

blocks['population'] = blocks['population'].fillna(0)
blocks['block_id'] = blocks.index.astype(int)

blocks.to_crs(4326).to_file(DATA_DIR / 'blocks.geojson', driver='GeoJSON')

print('Кварталов в Павловске:', len(blocks))
print('Суммарная площадь, га:', round(blocks['site_area'].sum() / 1e4, 1))
print('Оценка населения, чел.:', int(blocks['population'].sum()))
print('Ёмкость школ, мест:', int(blocks['capacity_school'].sum()))

ax = blocks.plot(figsize=(10, 10), color='#dddddd', edgecolor='white')
blocks.plot(column='land_use', categorical=True, legend=True, ax=ax, cmap='tab10')
ax.set_title(f'Кварталы Павловска: {len(blocks)} шт. (цвет — назначение территории)')
ax.set_axis_off()
plt.show()

---
## ⛔ ОСТАНОВКА после подготовки кварталов

Кварталы Павловска загружены и показаны выше. По плану здесь согласуется список показателей.

**Решение уже принято:** считаем все показатели из pipeline.ipynb — плотность, морфотипы, доступность территории и обеспеченность школами. Поэтому флаг CONFIRMED_TO_CONTINUE установлен в True и расчёты продолжаются.

Если нужно изменить набор показателей, поставьте CONFIRMED_TO_CONTINUE = False в ячейке настроек — предохранитель ниже остановит расчёты.


In [ ]:
assert CONFIRMED_TO_CONTINUE, (
    'ОСТАНОВКА. Сначала согласуйте список показателей и поставьте CONFIRMED_TO_CONTINUE = True.'
)

### Шаг 5. Время в пути между кварталами (готовый граф)

**Источник:** тот же официальный архив BlocksNet — файл `graph_drive.pickle`.

**Режим передвижения:** автомобиль (`drive`). **Единица времени:** минуты (атрибут рёбер `time_min`).

Граф построен для всего Санкт-Петербурга; Павловск находится внутри, поэтому граф подходит. Никакие графы (в том числе интермодальные) здесь не строятся.

In [ ]:
with open(graph_path, 'rb') as f:
    graph = pickle.load(f)

print('Тип объекта графа:', type(graph).__name__)
assert isinstance(graph, nx.Graph), 'Ожидался граф networkx'

# BlocksNet требует проекцию графа в виде целого кода EPSG
from pyproj import CRS

graph_crs = graph.graph.get('crs')
if graph_crs is not None and not isinstance(graph_crs, int):
    graph.graph['crs'] = CRS.from_user_input(graph_crs).to_epsg()
print('Проекция графа (EPSG):', graph.graph.get('crs'))
assert isinstance(graph.graph.get('crs'), int), 'Не удалось определить EPSG-код проекции графа'

# Матрица доступности считается по ориентированному графу
if not graph.is_directed():
    print('Граф неориентированный — дублируем рёбра в обе стороны')
    graph = graph.to_directed()

print('Узлов:', graph.number_of_nodes(), '| Рёбер:', graph.number_of_edges())

u, v, edge_data = next(iter(graph.edges(data=True)))
print('Пример ребра:', {k: edge_data[k] for k in list(edge_data)[:6]})
assert 'time_min' in edge_data, 'В рёбрах нет времени в минутах (time_min)'
print('Единица времени: минуты (time_min)')


Считаем матрицу времени в пути между кварталами Павловска. Значение в таблице — время от одного квартала до другого **в минутах**; для недостижимых пар — `inf`.

Матрица сохраняется в `accessibility_matrix_drive.csv` и переиспользуется при следующем запуске.

In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

matrix_path = DATA_DIR / 'accessibility_matrix_drive.csv'
acc_mx = None

if matrix_path.exists() and not FORCE_RELOAD:
    try:
        cached = pd.read_csv(matrix_path, index_col=0)
        cached.index = cached.index.astype(int)
        cached.columns = cached.columns.astype(int)
        if cached.shape == (len(blocks), len(blocks)) and set(cached.index) == set(blocks['block_id']):
            acc_mx = cached
            print('[из файла] матрица времени в пути прочитана')
        else:
            print('[!] сохранённая матрица не соответствует кварталам — пересчитываем')
    except Exception as error:
        print('[!] не удалось прочитать матрицу — пересчитываем:', type(error).__name__)

if acc_mx is None:
    graph_epsg = graph.graph.get('crs')
    blocks_for_matrix = blocks if graph_epsg == crs.to_epsg() else blocks.to_crs(graph_epsg)
    if graph_epsg != crs.to_epsg():
        print('Кварталы переведены в проекцию графа:', graph_epsg)
    print('Расчёт матрицы времени в пути... это может занять несколько минут')
    acc_mx = calculate_accessibility_matrix(blocks_for_matrix, graph, weight_key='time_min', dtype=np.float32)
    acc_mx.round(2).to_csv(matrix_path)
    print('[создано] матрица времени в пути сохранена')

values = acc_mx.to_numpy()
print('Размер матрицы:', acc_mx.shape)
print('Медиана времени в пути, мин:', round(float(np.median(values)), 2))
print('Доля недостижимых пар:', round(float(np.isinf(values).mean()), 4))


### Шаг 6. Показатели

- **Плотность** (`calculate_density_indicators`): FSI, GSI, MXI, L, OSR и доли жилой/нежилой площади.
- **Морфотипы** — Spacematrix и Strelka.
- **Доступность территории** (`area_accessibility`) — среднее время в пути из квартала, минуты.
- **Обеспеченность школами** (`competitive_provision`) — по нормативу BlocksNet.

> В версии 1.0.0.a9 функция `area_accessibility` находится в `blocksnet.analysis.network.accessibility` (в онлайн-примере путь другой — версии отличаются).

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

density_df = calculate_density_indicators(blocks)
DENSITY_COLUMNS = ['fsi', 'gsi', 'mxi', 'l', 'osr', 'share_living', 'share_non_living']
density_table = density_df[DENSITY_COLUMNS]
density_table.to_csv(DATA_DIR / 'density_indicators.csv')

density_table.describe().round(3)

In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
strelka_df = get_strelka_morphotypes(density_df)

spacematrix_table = spacematrix_df[['cluster', 'morphotype']].rename(columns={'morphotype': 'spacematrix_morphotype'})
strelka_table = strelka_df[['morphotype']].rename(columns={'morphotype': 'strelka_morphotype'})

spacematrix_table.to_csv(DATA_DIR / 'spacematrix.csv')
strelka_table.to_csv(DATA_DIR / 'strelka_morphotypes.csv')

print('Морфотипы Strelka:')
print(strelka_table['strelka_morphotype'].value_counts(dropna=False))

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

area_acc_df = area_accessibility(acc_mx, blocks)
area_acc_df.to_csv(DATA_DIR / 'area_accessibility.csv')
area_acc_df.head()

In [ ]:
from blocksnet.analysis.provision import competitive_provision
from blocksnet.config import service_types_config

school_cfg = service_types_config['school']
print('Норматив по школам:', school_cfg)

# Функция ожидает колонки population и capacity (целые числа)
provision_input = pd.DataFrame(
    {
        'population': blocks['population'].astype(float),
        'capacity': blocks['capacity_school'].fillna(0).round().astype(int),
    },
    index=blocks.index,
)

provision_df, provision_links_df = competitive_provision(
    provision_input,
    acc_mx,
    accessibility=school_cfg['accessibility'],
    demand=school_cfg['demand'],
)
provision_df.to_csv(DATA_DIR / 'provision_schools.csv')
provision_links_df.to_csv(DATA_DIR / 'provision_links_schools.csv')
provision_df.head()

Три карты с ключевыми показателями: плотность застройки (FSI), доступность территории (минуты) и обеспеченность школами.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

density_plot = blocks.join(density_table)
density_plot.plot(column='fsi', ax=axes[0], legend=True, cmap='viridis')
axes[0].set_title('FSI (плотность застройки)')

area_plot = blocks.join(area_acc_df['area_accessibility'])
area_plot['area_accessibility'] = area_plot['area_accessibility'].replace([np.inf, -np.inf], np.nan)
area_plot.plot(column='area_accessibility', ax=axes[1], legend=True, cmap='plasma')
axes[1].set_title('Доступность территории, мин')

blocks.join(provision_df['provision_strong']).plot(
    column='provision_strong', ax=axes[2], legend=True, cmap='RdYlGn', vmin=0, vmax=1
)
axes[2].set_title('Обеспеченность школами')

for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

### Шаг 7. Сохранение результатов

- `blocks_indicators.geojson` — кварталы со всеми показателями (для карт и ГИС);
- `blocks_indicators.csv` — таблица без геометрии (для Excel);
- `study_info.json` — паспорт исследования: источники, режим передвижения, версии.

In [ ]:
result = blocks.copy()
result = result.join(density_table)
result = result.join(strelka_table)
result = result.join(spacematrix_table)
result = result.join(area_acc_df['area_accessibility'])
result = result.join(provision_df[['provision_strong']])
result['travel_time_median_min'] = acc_mx.replace(np.inf, np.nan).median(axis=1)

# GeoJSON не поддерживает бесконечность: заменяем её на пропуск
numeric_columns = result.select_dtypes(include=[np.number]).columns
result[numeric_columns] = result[numeric_columns].replace([np.inf, -np.inf], np.nan)

result.to_crs(4326).to_file(DATA_DIR / 'blocks_indicators.geojson', driver='GeoJSON')
result.drop(columns='geometry').to_csv(DATA_DIR / 'blocks_indicators.csv')

info = {
    'territory': 'Павловск, Пушкинский район, Санкт-Петербург, Россия',
    'osm_relation_id': OSM_RELATION_ID,
    'data_source': 'IDUclub/blocksnet-data v0.0.1 (saint_petersburg.zip)',
    'blocks_source': 'готовые кварталы BlocksNet, отфильтрованы по границе Павловска',
    'travel_mode': 'drive (автомобиль)',
    'travel_time_units': 'minutes',
    'graph_source': DATASET_URL,
    'crs_for_calculations': str(crs),
    'blocks': int(len(result)),
    'blocksnet_version': blocksnet.__version__,
    'iduedu_version': iduedu.__version__,
}
(DATA_DIR / 'study_info.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')

print('Сохранено:')
print(' - blocks_indicators.geojson')
print(' - blocks_indicators.csv')
print(' - study_info.json')
result.head()

### Шаг 8. Проверка результатов

Автоматические проверки: кварталы не пусты, номера уникальны, матрица времени совпадает по размеру с числом кварталов, времена не отрицательные, геометрии корректны, файлы сохранены.

In [ ]:
checks = []
checks.append(('Кварталы не пусты', len(result) > 0))
checks.append(('Номера кварталов уникальны', bool(result['block_id'].is_unique)))
checks.append(('Матрица времени совпадает по размеру', acc_mx.shape == (len(blocks), len(blocks))))
checks.append(('Индексы кварталов и матрицы совпадают', set(acc_mx.index) == set(blocks['block_id'])))

finite_values = acc_mx.to_numpy()
finite_values = finite_values[np.isfinite(finite_values)]
checks.append(('Времена в пути не отрицательные', bool((finite_values >= 0).all())))
checks.append(('Нет пропусков в населении', bool(result['population'].notna().all())))
checks.append(('Геометрии корректны', bool(result.geometry.is_valid.all())))
checks.append(('Сохранён GeoJSON с показателями', (DATA_DIR / 'blocks_indicators.geojson').exists()))

for name, ok in checks:
    print(('OK      ' if ok else 'ОШИБКА !') + ' | ' + name)

assert all(ok for _, ok in checks), 'Не все проверки пройдены'

## Повторный запуск и что дальше

**Повторный запуск:** меню `Run` → `Run All Cells`. Архив и результаты уже на диске, поэтому заново ничего не скачивается. Чтобы перезагрузить данные принудительно, поставьте `FORCE_RELOAD = True` в ячейке настроек.

**Удалить все данные:** удалите папку `pavlovsk_study_data`.

**Что смотреть в результатах:**
- `blocks_indicators.geojson` / `.csv` — главный результат;
- `accessibility_matrix_drive.csv` — время в пути между кварталами (минуты);
- `density_indicators.csv`, `spacematrix.csv`, `strelka_morphotypes.csv`, `area_accessibility.csv`, `provision_schools.csv` — отдельные показатели;
- `study_info.json` — источники и версии.

**Следующий этап (отдельно):** один HTML-файл с интерактивной картой и панелью — те же показатели, графики, таблицы и выводы. Данные берутся из `pavlovsk_study_data`.